In [ ]:
import os
import re
import sys
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
from databricks import sql
from yfpy.query import YahooFantasySportsQuery
from dotenv import load_dotenv
from pathlib import Path

# Get the absolute path to the project root
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

from backend.models import FantasyLeague
from backend.analysis import matchup_analysis

load_dotenv()

True

In [2]:
with sql.connect(
    server_hostname=os.getenv("DATABRICKS_URL"),
    http_path=os.getenv("SQL_WAREHOUSE_HTTP_PATH"),
    access_token=os.getenv("BACKEND_TOKEN")
    ) as connection:
        with connection.cursor() as cursor:
            
            # fetch and cache NBA Schedule
            cursor.execute("SELECT * FROM nba_fantasy.gold.nba_games_schedule")
            arrow_table = cursor.fetchall_arrow()
            nba_schedule = arrow_table.to_pandas()

            player_stats = {}
            # fetch and cache player stats/rankings
            for table in ["ema", "ros_rankings", "weekly_rankings", "preseason_rankings"]:
                cursor.execute(f"SELECT * FROM nba_fantasy.gold.{table}")
                arrow_table = cursor.fetchall_arrow()
                df = arrow_table.to_pandas()
                player_stats[table] = (
                    df.replace({np.nan: None})
                      .set_index('PLAYER_NAME')
                      .to_dict(orient='index')
                )

In [3]:
auth_path = Path("../yahoo_auth")
yahoo_query = YahooFantasySportsQuery(
    league_id="66811",
    game_code="nba",
    offline=False,
    yahoo_consumer_key=os.getenv("YAHOO_CONSUMER_KEY"),
    yahoo_consumer_secret=os.getenv("YAHOO_CONSUMER_SECRET"),
    env_file_location= auth_path,
    save_token_data_to_env_file=True
)

In [4]:
league_id = 66811
team_id = 3

league = FantasyLeague(yahoo_query, player_stats)
league_dict = league.to_dict()

2026-07-07 11:29:14.532 - WARNING - query.py - yfpy.query:1029 - No game id or season/year provided, defaulting to current fantasy season.
2026-07-07 11:29:17.265 - WARNING - query.py - yfpy.query:1029 - No game id or season/year provided, defaulting to current fantasy season.
2026-07-07 11:29:19.661 - WARNING - query.py - yfpy.query:1029 - No game id or season/year provided, defaulting to current fantasy season.
2026-07-07 11:29:20.853 - WARNING - query.py - yfpy.query:1029 - No game id or season/year provided, defaulting to current fantasy season.
2026-07-07 11:29:22.005 - WARNING - query.py - yfpy.query:1029 - No game id or season/year provided, defaulting to current fantasy season.
2026-07-07 11:29:23.156 - WARNING - query.py - yfpy.query:1029 - No game id or season/year provided, defaulting to current fantasy season.
2026-07-07 11:29:24.373 - WARNING - query.py - yfpy.query:1029 - No game id or season/year provided, defaulting to current fantasy season.
2026-07-07 11:29:25.591 - W

In [5]:
team1 = league_dict["teams"][2]
team2 = league_dict["teams"][1]
week_num = 3
date_start = "2026-03-16"
roster_positions = league_dict["roster_positions"]
stats_source = "weekly_stats"
nba_schedule = nba_schedule

yahoo_query = YahooFantasySportsQuery(
    league_id=league_id,
    game_code="nba",
    offline=False,
    yahoo_consumer_key=os.getenv("YAHOO_CONSUMER_KEY"),
    yahoo_consumer_secret=os.getenv("YAHOO_CONSUMER_SECRET"),
    env_file_location= Path("/app/auth"),
    save_token_data_to_env_file=False
)

matchups_analysis = matchup_analysis(yahoo_query, team1, team2, week_num, date_start, roster_positions, stats_source, nba_schedule)


2026-07-07 11:29:34.061 - WARNING - query.py - yfpy.query:1029 - No game id or season/year provided, defaulting to current fantasy season.
2026-07-07 11:29:34.497 - WARNING - query.py - yfpy.query:1029 - No game id or season/year provided, defaulting to current fantasy season.


In [6]:
rostered_players = set()
for team in league.teams:
    for player in team.players:
        rostered_players.add(player.name)

In [7]:
len(rostered_players)

160

In [8]:
fa = []
for player, stats in player_stats["weekly_rankings"].items():
    if player not in rostered_players:
        fa.append((player, stats))

In [9]:
len(fa)

374